# Classical ML Models — Rain in Australia

This notebook trains and evaluates four classical ML models for predicting **RainTomorrow** (binary):
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. XGBoost

All model pipelines are defined in `classical_models.py`.  
Tuning & calibration utilities are in `hyperparameter_tuning.py` and used in `classical_finetuning.ipynb`.  
Structured logging is provided by `training_logger.py`.

**Primary evaluation metric:** ROC-AUC (per cahier de charge).

## 1 — Imports & Constants

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import RocCurveDisplay
from classical_models import (
    get_all_models, evaluate_model,
    FEATURE_COLUMNS, TARGET_RAIN, LOCATION_COLUMN, DATA_PATH,
)
from training_logger import setup_logger, log_training_metrics, log_cv_results, log_model_save

from datetime import datetime
from tqdm.auto import tqdm

MODELS_DIR  = "saved_models/"
os.makedirs(MODELS_DIR, exist_ok=True)

# Setup structured logger
logger = setup_logger("classical_training")
logger.info("=== Classical Training Session Started ===")

def ts():
    return datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

print("Imports OK")

/home/limam/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
02:15:24 | INFO | === Classical Training Session Started ===


Imports OK


## 2 — Data Loading

In [2]:
# Load the feature-engineered dataset
df = pd.read_csv(DATA_PATH)

# Schema validation
print("Columns:", df.columns.tolist())
print(f"Shape: {df.shape}")

# Extract features (excludes Location string column) and target
X = df[FEATURE_COLUMNS]
y = df[TARGET_RAIN]
print(f"\nFeature matrix: {X.shape}")
print(f"Rain prevalence: {y.mean():.2%}")
print(f"Locations available: {df[LOCATION_COLUMN].nunique()}")

logger.info(f"Data loaded: {X.shape[0]} rows, {X.shape[1]} features, rain={y.mean():.2%}")

02:15:25 | INFO | Data loaded: 142193 rows, 21 features, rain=22.42%


Columns: ['Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Temp9am', 'Temp3pm', 'Month', 'DayOfYear', 'WeekOfYear', 'Quarter', 'Month_sin', 'Month_cos', 'Day_sin', 'Day_cos', 'TempRange', 'TempMean', 'TempDiff_9_3', 'WindChill9am', 'WindChill3pm', 'HumidityChange', 'HumidityMean', 'PressureChange', 'PressureMean', 'DewPoint9am', 'DewPoint3pm', 'HeatIndex9am', 'HeatIndex3pm', 'WindDirGust_sin', 'WindDirGust_cos', 'WindDir9am_sin', 'WindDir9am_cos', 'WindDir3pm_sin', 'WindDir3pm_cos', 'WindSpeedChange', 'WindSpeedMean', 'StormFlag', 'RainToday_bin', 'RainRiskScore', 'TempHumidity_interact', 'Rainfall_lag1', 'Rainfall_lag2', 'Rainfall_lag7', 'Rainfall_roll3', 'Rainfall_roll7', 'Humidity3pm_lag1', 'Humidity3pm_lag2', 'Humidity3pm_lag7', 'Humidity3pm_roll3', 'Humidity3pm_roll7', 'MaxTemp_lag1', 'MaxTemp_lag2', 'MaxTemp_lag7', 'MaxTemp_roll3', 'MaxTemp_roll7', 'MinTemp_lag1'

## 3 — Train / Test Split

In [3]:
# Stratified split to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y      # Critical: preserve ~22% rain ratio
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Rain prevalence — Train: {y_train.mean():.2%}, Test: {y_test.mean():.2%}")

logger.info(f"Train/test split: train={X_train.shape}, test={X_test.shape}")

02:15:25 | INFO | Train/test split: train=(113754, 21), test=(28439, 21)


Train: (113754, 21), Test: (28439, 21)
Rain prevalence — Train: 22.42%, Test: 22.42%


## 4 — Model Instantiation Check

In [4]:
# Verify all 4 models instantiate without error
models = get_all_models()
for name, pipeline in models.items():
    clf_name = type(pipeline.steps[-1][1]).__name__
    print(f"✅ {name}: {clf_name}")
    logger.info(f"Model instantiated: {name} ({clf_name})")

02:15:25 | INFO | Model instantiated: logistic_regression (LogisticRegression)
02:15:25 | INFO | Model instantiated: decision_tree (DecisionTreeClassifier)
02:15:25 | INFO | Model instantiated: random_forest (RandomForestClassifier)
02:15:25 | INFO | Model instantiated: xgboost (XGBClassifier)


✅ logistic_regression: LogisticRegression
✅ decision_tree: DecisionTreeClassifier
✅ random_forest: RandomForestClassifier
✅ xgboost: XGBClassifier


## 5 — Cross-Validation

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["roc_auc", "f1", "accuracy", "precision", "recall"]

cv_results = {}
for name, pipeline in tqdm(models.items(), desc="Cross-Validation", unit="model"):
    print(f"\nRunning CV for {name}...")
    scores = cross_validate(
        pipeline, X_train, y_train,
        cv=cv_strategy, scoring=scoring,
        n_jobs=-1, return_train_score=True,
        verbose=1
    )
    cv_results[name] = scores
    print(f"  ROC-AUC: {scores["test_roc_auc"].mean():.4f} ± {scores["test_roc_auc"].std():.4f}")
    print(f"  F1:      {scores["test_f1"].mean():.4f} ± {scores["test_f1"].std():.4f}")
    print(f"  Accuracy:{scores["test_accuracy"].mean():.4f} ± {scores["test_accuracy"].std():.4f}")
    
    # Structured logging
    log_cv_results(logger, name, scores)

print("\n✅ Cross-validation complete for all models.")

## 6 — Model Training & Evaluation

Train all models on the full training set and evaluate on the test set.

In [ ]:
for name, pipeline in tqdm(models.items(), desc="Training models", unit="model"):
    print(f"\n⏳ Training {name}...")
    logger.info(f"Training {name}...")
    pipeline.fit(X_train, y_train)
    print(f"  ✅ {name} fitted.")
    
    # Evaluate on test set
    metrics = evaluate_model(pipeline, X_test, y_test)
    print(f"  Accuracy:  {metrics["accuracy"]:.4f}")
    print(f"  ROC-AUC:   {metrics["roc_auc"]:.4f}")
    print(f"  F1:        {metrics["f1_score"]:.4f}")
    print(f"  Precision: {metrics["precision"]:.4f}")
    print(f"  Recall:    {metrics["recall"]:.4f}")
    
    # Structured logging
    log_training_metrics(logger, name, metrics, phase="evaluation")
    
    # Save model
    model_path = os.path.join(MODELS_DIR, f"{name}.joblib")
    joblib.dump(pipeline, model_path)
    print(f"  Saved → {model_path}")
    log_model_save(logger, name, model_path)

print("\n✅ All models trained and saved!")

## 7 — ROC Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for ax, (name, pipeline) in zip(axes, models.items()):
    RocCurveDisplay.from_estimator(pipeline, X_test, y_test, ax=ax, name=name)
    ax.set_title(f"{name}")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("artifacts/roc_curves_classical.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ ROC curves saved to artifacts/roc_curves_classical.png")

## 8 — Results Summary Table

In [ ]:
summary_rows = []
for name, pipeline in models.items():
    metrics = evaluate_model(pipeline, X_test, y_test)
    summary_rows.append({"Model": name, **metrics})

summary_df = pd.DataFrame(summary_rows).set_index("Model")
summary_df = summary_df.drop(columns=["confusion_matrix"])
print(summary_df.sort_values("roc_auc", ascending=False).to_markdown())

## 9 — Temperature Regressor Training

Train a **GradientBoostingRegressor** pipeline to predict `MaxTemp`.

**Important:** `MaxTemp` is in `FEATURE_COLUMNS` — we remove it from X when predicting it to avoid data leakage.

In [ ]:
from classical_models import build_temp_regressor, evaluate_regressor, TARGET_TEMP

# --- Prepare temperature features (remove MaxTemp to avoid leakage) ---
temp_features = [c for c in FEATURE_COLUMNS if c != "MaxTemp"]

# Drop rows where MaxTemp is NaN
df_temp = df.dropna(subset=["MaxTemp"]).copy()

X_temp = df_temp[temp_features]
y_temp = df_temp[TARGET_TEMP]

# --- Chronological 80/20 split (no shuffle — preserve temporal order) ---
split_idx = int(len(X_temp) * 0.8)
X_temp_train, X_temp_test = X_temp.iloc[:split_idx], X_temp.iloc[split_idx:]
y_temp_train, y_temp_test = y_temp.iloc[:split_idx], y_temp.iloc[split_idx:]

print(f"Temp Train: {X_temp_train.shape}, Temp Test: {X_temp_test.shape}")
print(f"MaxTemp range — Train: [{y_temp_train.min():.1f}, {y_temp_train.max():.1f}], "
      f"Test: [{y_temp_test.min():.1f}, {y_temp_test.max():.1f}]")

logger.info(f"Temp regressor data: train={X_temp_train.shape}, test={X_temp_test.shape}")

## 10 — Temperature Regressor Evaluation & Save

In [ ]:
# --- Build and fit the temperature regressor ---
temp_model = build_temp_regressor()
print("⏳ Fitting temperature regressor...")
logger.info("Fitting temperature regressor...")
temp_model.fit(X_temp_train, y_temp_train)
print("✅ Temperature regressor fitted.")

# --- Evaluate ---
temp_metrics = evaluate_regressor(temp_model, X_temp_test, y_temp_test)
print(f"Temperature Regressor Results:")
print(f"  MAE : {temp_metrics["mae"]:.4f}")
print(f"  R²  : {temp_metrics["r2"]:.4f}")

log_training_metrics(logger, "temp_regressor", temp_metrics, phase="evaluation")

# --- Save ---
temp_model_path = os.path.join(MODELS_DIR, "temp_regressor.joblib")
joblib.dump(temp_model, temp_model_path)
print(f"\nSaved temperature regressor → {temp_model_path}")
log_model_save(logger, "temp_regressor", temp_model_path)